# CartPole Q-Learning Implementation
**Author:** Vinol  
**Course:** Scalable Computing - Trinity College Dublin  
**Date:** November 2025

---

## Overview
This notebook implements Q-Learning to solve the CartPole-v1 environment from OpenAI Gymnasium.

**Objective:** Train an agent to balance a pole on a moving cart using Q-Learning with discretized state space.

## 1. Setup and Installation

In [ ]:
!pip install gymnasium matplotlib numpy -q

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

print("Libraries imported successfully!")
print(f"Gymnasium version: {gym.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Environment Setup

In [ ]:
env = gym.make('CartPole-v1')

print("="*60)
print("CARTPOLE ENVIRONMENT")
print("="*60)
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Number of actions: {env.action_space.n}")
print()

state, info = env.reset(seed=42)
print("Sample initial state:", state)
print("State components:")
print(f"  [0] Cart Position: {state[0]:.4f}")
print(f"  [1] Cart Velocity: {state[1]:.4f}")
print(f"  [2] Pole Angle: {state[2]:.4f} radians ({np.degrees(state[2]):.2f}°)")
print(f"  [3] Pole Angular Velocity: {state[3]:.4f}")
print("="*60)

## 3. Discretization Configuration

In [ ]:
n_bins = [8, 10, 20, 20]
state_bounds = [
    [-2.4, 2.4],
    [-3, 3],
    [-0.3, 0.3],
    [-3, 3]
]

def create_bins(n_bins, state_bounds):
    bins = []
    for i in range(len(n_bins)):
        bins.append(np.linspace(state_bounds[i][0], state_bounds[i][1], n_bins[i]))
    return bins

def discretize_state(state, bins):
    discrete_state = []
    for i in range(len(state)):
        discrete_state.append(np.digitize(state[i], bins[i]) - 1)
    return tuple(discrete_state)

bins = create_bins(n_bins, state_bounds)

print("Discretization Configuration:")
print(f"  Bins per dimension: {n_bins}")
print(f"  Total discrete states: {np.prod(n_bins):,}")
print(f"  State bounds: {state_bounds}")
print()

test_state = [1.5, -0.5, 0.1, 0.3]
print(f"Example: Continuous {test_state} → Discrete {discretize_state(test_state, bins)}")

## 4. Q-Table Initialization

In [ ]:
q_table_shape = n_bins + [env.action_space.n]
q_table = np.zeros(q_table_shape)

print(f"Q-Table shape: {q_table.shape}")
print(f"Total Q-values: {np.prod(q_table_shape):,}")
print(f"Memory size: ~{q_table.nbytes / 1024:.2f} KB")

## 5. Hyperparameters

In [ ]:
learning_rate = 0.15
discount_factor = 0.99
epsilon = 1.0
epsilon_decay = 0.9995
epsilon_min = 0.01
n_episodes = 15000
max_steps = 500
solved_threshold = 195
stop_when_solved = True

print("Hyperparameters:")
print(f"  Learning rate (α): {learning_rate}")
print(f"  Discount factor (γ): {discount_factor}")
print(f"  Initial epsilon (ε): {epsilon}")
print(f"  Epsilon decay: {epsilon_decay}")
print(f"  Minimum epsilon: {epsilon_min}")
print(f"  Max episodes: {n_episodes:,}")
print(f"  Max steps per episode: {max_steps}")
print(f"  Solved threshold: {solved_threshold}")

## 6. Core Functions

In [ ]:
def choose_action(state, epsilon):
    if np.random.random() < epsilon:
        return env.action_space.sample()
    else:
        return np.argmax(q_table[state])

def update_q_table(state, action, reward, next_state, done):
    current_q = q_table[state + (action,)]
    max_next_q = 0 if done else np.max(q_table[next_state])
    new_q = current_q + learning_rate * (reward + discount_factor * max_next_q - current_q)
    q_table[state + (action,)] = new_q

print("Core functions defined:")
print("  ✓ choose_action(state, epsilon)")
print("  ✓ update_q_table(state, action, reward, next_state, done)")

## 7. Training Loop

In [ ]:
episode_rewards = []
epsilon_history = []

print("="*60)
print("STARTING TRAINING")
print("="*60)

start_time = time.time()

for episode in range(n_episodes):
    state, info = env.reset()
    state = discretize_state(state, bins)
    total_reward = 0
    
    for step in range(max_steps):
        action = choose_action(state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        next_state = discretize_state(next_state, bins)
        done = terminated or truncated
        
        update_q_table(state, action, reward, next_state, done)
        
        state = next_state
        total_reward += reward
        
        if done:
            break
    
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    episode_rewards.append(total_reward)
    epsilon_history.append(epsilon)
    
    if stop_when_solved and len(episode_rewards) >= 100:
        recent_avg = np.mean(episode_rewards[-100:])
        if recent_avg >= solved_threshold:
            print(f"\n🎉 CartPole SOLVED at episode {episode + 1}!")
            print(f"Average reward: {recent_avg:.2f}")
            print(f"Training time: {time.time() - start_time:.2f} seconds")
            break
    
    if (episode + 1) % 100 == 0:
        avg_reward = np.mean(episode_rewards[-100:])
        print(f"Episode {episode + 1:5d}/{n_episodes} | Avg: {avg_reward:6.2f} | Epsilon: {epsilon:.3f} | Last: {total_reward:3.0f}")
        
        # Live update plot every 500 episodes
        if (episode + 1) % 500 == 0:
            clear_output(wait=True)
            plt.figure(figsize=(12, 4))
            plt.subplot(1, 2, 1)
            plt.plot(episode_rewards, alpha=0.6)
            plt.axhline(y=195, color='r', linestyle='--')
            plt.xlabel('Episode')
            plt.ylabel('Reward')
            plt.title('Training Progress')
            
            plt.subplot(1, 2, 2)
            if len(episode_rewards) >= 100:
                ma = np.convolve(episode_rewards, np.ones(100)/100, mode='valid')
                plt.plot(ma, color='green')
                plt.axhline(y=195, color='r', linestyle='--')
            plt.xlabel('Episode')
            plt.ylabel('100-Episode Average')
            plt.title('Moving Average')
            plt.tight_layout()
            plt.show()
            
            print(f"Episode {episode + 1:5d}/{n_episodes} | Avg: {avg_reward:6.2f} | Epsilon: {epsilon:.3f}")

training_time = time.time() - start_time
print("="*60)
print(f"TRAINING COMPLETED in {training_time:.2f} seconds")
print("="*60)

## 8. Results Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Raw rewards
axes[0, 0].plot(episode_rewards, alpha=0.6, color='blue', linewidth=0.5)
axes[0, 0].axhline(y=195, color='red', linestyle='--', linewidth=2, label='Solved (195)')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')
axes[0, 0].set_title('Raw Rewards per Episode')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Moving average
window = 100
moving_avg = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
axes[0, 1].plot(moving_avg, color='green', linewidth=2)
axes[0, 1].axhline(y=195, color='red', linestyle='--', linewidth=2, label='Solved (195)')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Average Reward')
axes[0, 1].set_title(f'Moving Average (window={window})')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Epsilon decay
axes[1, 0].plot(epsilon_history, color='orange', linewidth=2)
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Epsilon')
axes[1, 0].set_title('Exploration Rate (Epsilon) Decay')
axes[1, 0].grid(True, alpha=0.3)

# Statistics
stats_text = f"""
TRAINING SUMMARY
================

Configuration:
• Bins: {n_bins}
• Learning Rate: {learning_rate}
• Discount Factor: {discount_factor}
• Epsilon Decay: {epsilon_decay}

Results:
• Total Episodes: {len(episode_rewards)}
• Training Time: {training_time:.2f}s
• Final Epsilon: {epsilon_history[-1]:.4f}

Performance:
• First 100 avg: {np.mean(episode_rewards[:100]):.2f}
• Last 100 avg: {np.mean(episode_rewards[-100:]):.2f}
• Max episode: {np.max(episode_rewards):.0f}
• Min episode: {np.min(episode_rewards):.0f}

Success:
• Episodes ≥195: {np.sum(np.array(episode_rewards) >= 195)}
• Success rate: {100*np.sum(np.array(episode_rewards) >= 195)/len(episode_rewards):.1f}%
"""

axes[1, 1].text(0.1, 0.5, stats_text, transform=axes[1, 1].transAxes,
                fontsize=10, verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 1].axis('off')
axes[1, 1].set_title('Training Statistics')

plt.suptitle('CartPole Q-Learning: Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Test Trained Agent

In [ ]:
print("Testing trained agent (epsilon=0, pure exploitation)...\n")

test_rewards = []
n_test_episodes = 10

for episode in range(n_test_episodes):
    state, info = env.reset()
    state = discretize_state(state, bins)
    total_reward = 0
    
    for step in range(max_steps):
        action = np.argmax(q_table[state])
        next_state, reward, terminated, truncated, info = env.step(action)
        next_state = discretize_state(next_state, bins)
        state = next_state
        total_reward += reward
        
        if terminated or truncated:
            break
    
    test_rewards.append(total_reward)
    print(f"  Test Episode {episode + 1:2d}: {total_reward:3.0f} steps")

avg_test = np.mean(test_rewards)
std_test = np.std(test_rewards)

print(f"\nTest Results:")
print(f"  Average: {avg_test:.2f} ± {std_test:.2f}")
print(f"  Min: {np.min(test_rewards):.0f}")
print(f"  Max: {np.max(test_rewards):.0f}")

if avg_test >= 195:
    print(f"\n  ✓ Agent SOLVED CartPole!")
else:
    print(f"\n  ✗ Agent needs improvement (target: 195+)")

env.close()

## 10. Additional Analysis (Optional)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Reward distribution
axes[0].hist(episode_rewards, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(x=195, color='red', linestyle='--', linewidth=2, label='Solved (195)')
axes[0].set_xlabel('Reward')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Episode Rewards')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Q-value distribution
q_values_left = []
q_values_right = []
for state_idx in range(min(1000, np.prod(n_bins))):
    indices = np.unravel_index(state_idx, n_bins)
    q_values_left.append(q_table[indices + (0,)])
    q_values_right.append(q_table[indices + (1,)])

axes[1].hist(q_values_left, bins=50, alpha=0.6, label='Push Left', color='blue')
axes[1].hist(q_values_right, bins=50, alpha=0.6, label='Push Right', color='red')
axes[1].set_xlabel('Q-Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Q-Values by Action')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Success rate over time
window = 100
success_rate = []
for i in range(window, len(episode_rewards) + 1):
    success_rate.append(np.mean(np.array(episode_rewards[i-window:i]) >= 195) * 100)

axes[2].plot(range(window, len(episode_rewards) + 1), success_rate, color='purple', linewidth=2)
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Success Rate (%)')
axes[2].set_title(f'Success Rate (% episodes ≥195, window={window})')
axes[2].axhline(y=50, color='orange', linestyle='--', alpha=0.5, label='50%')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Visualize Agent (Optional)

**Note:** Rendering in Colab requires additional setup. Uncomment to try:

In [ ]:
# # Install rendering dependencies
# !apt-get install -y xvfb python-opengl > /dev/null 2>&1
# !pip install pyvirtualdisplay > /dev/null 2>&1

# from pyvirtualdisplay import Display
# from IPython.display import HTML
# from base64 import b64encode

# display = Display(visible=0, size=(400, 300))
# display.start()

# # Record a video
# env_visual = gym.make('CartPole-v1', render_mode='rgb_array')
# frames = []

# state, info = env_visual.reset()
# state = discretize_state(state, bins)

# for _ in range(500):
#     frames.append(env_visual.render())
#     action = np.argmax(q_table[state])
#     next_state, reward, terminated, truncated, info = env_visual.step(action)
#     state = discretize_state(next_state, bins)
#     if terminated or truncated:
#         break

# env_visual.close()
# print(f"Recorded {len(frames)} frames")

## Conclusion

This notebook successfully implemented Q-Learning for the CartPole-v1 environment.

**Key Results:**
- Agent learned to balance the pole through trial and error
- Used discretization to handle continuous state space
- Achieved solving threshold (195+ average reward)
- Demonstrated epsilon-greedy exploration strategy

**Next Steps:**
- Try different hyperparameters
- Experiment with bin configurations
- Compare with other RL algorithms (DQN, PPO)
- Apply to other Gymnasium environments